In [7]:
import os
import re
import json
from collections import Counter, defaultdict
from typing import List, Dict, Any, Optional

import pandas as pd
from IPython.display import display


In [8]:
ONE_LINE_INPUT_PATH = "./one_line_list.json"
IMAGE_BASE_URL = "http://3.35.185.251:8000"

INVALID_CONTENT_PATTERNS = [
    r"이미지가 제대로 보이지 않아",
    r"다시 한 번 이미지 업로드",
    r"일기를 작성하기 어려워요",
]


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\\s+", " ", str(text)).strip()


def build_full_image_url(image_url: Optional[str]) -> Optional[str]:
    if not image_url:
        return None
    image_url = str(image_url).strip()
    if image_url.startswith("http://") or image_url.startswith("https://"):
        return image_url
    return f"{IMAGE_BASE_URL}{image_url}"


with open(ONE_LINE_INPUT_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

rows = []
for user_block in raw:
    user_id = user_block.get("user_id")
    for diary in user_block.get("diaries", []):
        content = normalize_whitespace(diary.get("content", ""))
        if not content:
            continue
        if any(re.search(pattern, content) for pattern in INVALID_CONTENT_PATTERNS):
            continue

        rows.append({
            "user_id": user_id,
            "diary_id": diary.get("id"),
            "date": diary.get("diary_date"),
            "content": content,
            "image_url": diary.get("image_url"),
            "full_image_url": build_full_image_url(diary.get("image_url")),
            "created_at": diary.get("created_at")
        })

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)

print("총 한줄일기 수:", len(df))
print("사용자 수:", df["user_id"].nunique())
display(df.head())


총 한줄일기 수: 660
사용자 수: 6


,user_id,diary_id,date,content,image_url,full_image_url,created_at,month
0,4,16,2025-12-12,따뜻한 차 안에서 간식 먹는 행복 💕🍪🚗,/media/images/20251212_155610_740766d2.jpg,http://3.35.185.251:8000/media/images/20251212...,2025-12-12T15:56:10.957176,2025-12
1,4,17,2025-12-12,장난감 손에 들고 상상의 모험 떠나요🌟🔫🧙‍♂️,/media/images/20251212_155639_e5c6a750.jpg,http://3.35.185.251:8000/media/images/20251212...,2025-12-12T15:56:39.146318,2025-12
2,4,18,2025-12-12,고기 냄새에 반한 작은 미식가의 행복한 한 끼🍖💖,/media/images/20251212_155711_d6367e62.jpg,http://3.35.185.251:8000/media/images/20251212...,2025-12-12T15:57:11.014722,2025-12
3,4,19,2025-12-12,따뜻한 고기 굽는 불빛에 마음이 포근해져요🔥🥩❤️,/media/images/20251212_155731_d5e9e5ee.jpg,http://3.35.185.251:8000/media/images/20251212...,2025-12-12T15:57:31.413000,2025-12
4,4,20,2025-12-12,따뜻한 불꽃 아래서 손 꼭 잡았어요🔥🤝✨,/media/images/20251212_155751_43641256.jpg,http://3.35.185.251:8000/media/images/20251212...,2025-12-12T15:57:51.405639,2025-12


In [9]:
EMOTION_PATTERNS = [
    r"행복", r"기분", r"마음", r"포근", r"설레", r"웃음", r"사랑"
]

LOW_INFO_PATTERNS = [
    r"오늘", r"하루", r"순간", r"시간", r"가득", r"작은"
]

PLACE_LEXICON = {
    "차 안": "차 안",
    "차": "차 안",
    "자동차": "차 안",
    "집": "집",
    "공원": "공원",
    "놀이터": "놀이터",
    "식당": "식당",
    "눈밭": "눈밭"
}

COMPANION_LEXICON = {
    "아빠": "아빠",
    "엄마": "엄마",
    "친구": "친구",
    "가족": "가족",
    "선생님": "선생님",
    "형제": "형제"
}

OBJECT_LEXICON = {
    "간식": "간식",
    "과자": "간식",
    "쿠키": "간식",
    "고기": "고기",
    "불꽃": "불꽃",
    "장난감": "장난감",
    "총": "장난감 총",
    "블록": "블록",
    "책": "책",
    "곰인형": "곰인형",
    "인형": "인형",
    "눈": "눈",
    "썰매": "썰매"
}

ACTION_RULES = [
    (r"(먹|간식|과자|쿠키|한 끼)", "snack_eating", "간식/먹기", "식사/간식"),
    (r"(고기|굽)", "meal_time", "식사 시간", "식사/간식"),
    (r"(손 꼭 잡|손 잡)", "hold_hands", "손 잡기", "가족상호작용"),
    (r"(장난감|모험|놀이)", "play_time", "놀이", "놀이"),
    (r"(블록)", "block_play", "블록 놀이", "만들기"),
    (r"(책|읽)", "reading", "책 읽기", "기관생활"),
    (r"(눈|썰매)", "snow_play", "눈 놀이", "바깥활동"),
    (r"(걷|산책)", "walk", "산책", "바깥활동"),
    (r"(안|안아|품에)", "hugging", "안아주기", "가족상호작용")
]

FREE_KEYWORD_STOPWORDS = {
    "행복", "행복한", "따뜻한", "작은", "달콤한", "가득", "가득한", "시간",
    "마음", "오늘", "하루", "정말", "반한", "포근해져요", "피었어요"
}


In [10]:
def split_sentences(text: str) -> List[str]:
    text = str(text).strip()
    if not text:
        return []
    sents = re.split(r"(?<=[.!?다요])\s+", text)
    return [s.strip() for s in sents if s.strip()]


def soft_clean_sentence(sent: str) -> str:
    s = sent
    for p in EMOTION_PATTERNS:
        s = re.sub(p, " ", s)
    for p in LOW_INFO_PATTERNS:
        s = re.sub(p, " ", s)
    s = re.sub(r"[\"'“”‘’]", " ", s)
    s = re.sub(r"\s+", " ", s).strip(" ,.")
    return normalize_whitespace(s)


def preprocess_diary(text: str) -> Dict[str, Any]:
    original_sents = split_sentences(text)
    cleaned_sents = []
    for sent in original_sents:
        cleaned = soft_clean_sentence(sent)
        if len(cleaned) >= 2:
            cleaned_sents.append(cleaned)
    return {
        "original_sentences": original_sents,
        "cleaned_sentences": cleaned_sents
    }


def detect_places(text: str) -> List[str]:
    return sorted({label for token, label in PLACE_LEXICON.items() if token in text})


def detect_companions(text: str) -> List[str]:
    return sorted({label for token, label in COMPANION_LEXICON.items() if token in text})


def detect_objects(text: str) -> List[str]:
    return sorted({label for token, label in OBJECT_LEXICON.items() if token in text})


def detect_actions(text: str) -> List[Dict[str, str]]:
    found = []
    for pattern, canonical, label, category in ACTION_RULES:
        if re.search(pattern, text):
            found.append({
                "raw_action": pattern,
                "canonical_action": canonical,
                "action_label": label,
                "category": category
            })
    unique = {}
    for item in found:
        unique[item["canonical_action"]] = item
    return list(unique.values())


def extract_free_keywords(text: str) -> List[str]:
    cleaned = re.sub(r"[\U00010000-\U0010ffff]", " ", text)
    cleaned = re.sub(r"[^\w\s가-힣]", " ", cleaned)
    cleaned = normalize_whitespace(cleaned)
    tokens = re.findall(r"[가-힣]{2,}", cleaned)
    out = []
    for token in tokens:
        if token in FREE_KEYWORD_STOPWORDS:
            continue
        out.append(token)
    return sorted(set(out))


def merge_unique_preserve_order(*groups: List[str]) -> List[str]:
    out = []
    seen = set()
    for group in groups:
        for item in group:
            item = str(item).strip()
            if not item or item in seen:
                continue
            seen.add(item)
            out.append(item)
    return out


def build_keyword_bundle(action_label: str,
                         places: List[str],
                         objects: List[str],
                         companions: List[str],
                         free_keywords: List[str]) -> Dict[str, Any]:
    keyword_types = {}
    ordered_keywords = []

    def add_keyword(keyword: str, keyword_type: str) -> None:
        if not keyword:
            return
        keyword = str(keyword).strip()
        if not keyword:
            return
        if keyword not in ordered_keywords:
            ordered_keywords.append(keyword)
        if keyword not in keyword_types or keyword_types[keyword] == "free":
            keyword_types[keyword] = keyword_type

    add_keyword(action_label, "action")
    for item in places:
        add_keyword(item, "place")
    for item in objects:
        add_keyword(item, "object")
    for item in companions:
        add_keyword(item, "companion")
    for item in free_keywords[:5]:
        add_keyword(item, "free")

    return {
        "photo_keywords": ordered_keywords,
        "photo_keyword_types": keyword_types
    }


def extract_rule_hints(text: str) -> Dict[str, Any]:
    return {
        "places": detect_places(text),
        "objects": detect_objects(text),
        "companions": detect_companions(text),
        "actions": detect_actions(text),
        "free_keywords": extract_free_keywords(text),
    }


def rule_based_scene_extract(row: pd.Series) -> List[Dict[str, Any]]:
    prep = preprocess_diary(row["content"])
    scenes = []

    for sent in prep["cleaned_sentences"] or [row["content"]]:
        actions = detect_actions(sent)
        places = detect_places(sent)
        companions = detect_companions(sent)
        objects = detect_objects(sent)

        if not actions and not places and not objects and not companions:
            actions = [{
                "raw_action": "",
                "canonical_action": "one_line_scene",
                "action_label": "한줄 기록",
                "category": "기타"
            }]

        free_keywords = extract_free_keywords(sent)

        for action in actions:
            keyword_bundle = build_keyword_bundle(
                action["action_label"],
                places,
                objects,
                companions,
                free_keywords,
            )

            scenes.append({
                "user_id": row["user_id"],
                "diary_id": row["diary_id"],
                "date": row["date"],
                "month": row["month"],
                "created_at": row["created_at"],
                "raw_text": sent,
                "content": row["content"],
                "canonical_action": action["canonical_action"],
                "action_label": action["action_label"],
                "category": action["category"],
                "places": places,
                "objects": objects,
                "companions": companions,
                "confidence": 0.45,
                "source_image_url": row["image_url"],
                "source_full_image_url": row["full_image_url"],
                "photo_keywords": keyword_bundle["photo_keywords"],
                "photo_keyword_types": keyword_bundle["photo_keyword_types"]
            })

    dedup = {}
    for sc in scenes:
        key = (
            sc["diary_id"],
            str(sc["date"]),
            sc["canonical_action"],
            tuple(sc["places"]),
            tuple(sc["objects"]),
            tuple(sc["companions"])
        )
        if key not in dedup:
            dedup[key] = sc

    return list(dedup.values())



In [ ]:
USE_GPT = True
OPENAI_API_KEY = YOUR_OPEN_API_KEY
OPENAI_MODEL = "gpt-4.1-mini"

GPT_SCENE_SYSTEM_PROMPT = """
너는 한줄 육아 기록에서 월간 리포트용 장면(scene)을 구조화하는 도우미다.

목표:
- 감정 표현과 수사보다 실제로 관찰 가능한 장면을 우선 추출한다.
- 없는 정보는 추측하지 않는다.
- 사진 UI 연결을 위해 장소, 사물, 함께한 사람을 최대한 보수적으로 남긴다.

반드시 JSON만 출력한다.
출력 형식:
{
  "scenes": [
    {
      "summary": "관찰 가능한 짧은 장면 설명",
      "canonical_action": "영문_스네이크케이스",
      "action_label": "사람이 읽을 한국어 행동 라벨",
      "category": "놀이|바깥활동|만들기|식사/간식|가족상호작용|역할놀이|기관생활|행사/이벤트|기타",
      "places": ["..."],
      "objects": ["..."],
      "companions": ["..."],
      "confidence": 0.0
    }
  ],
  "noise_expressions": ["감정/수식 표현"]
}
""".strip()


def extract_json_from_text(text: str) -> Dict[str, Any]:
    text = str(text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        return json.loads(m.group(0))

    raise ValueError("JSON 파싱 실패")


def build_gpt_scene_prompt(row: pd.Series) -> str:
    return f"""
사용자: {row['user_id']}
날짜: {str(row['date'].date())}
원문: {row['content']}

지시:
- 관찰 가능한 장면만 추출하라.
- 감정, 과장, 미사여구는 noise_expressions로 분리하라.
- 장소, 사물, 함께한 사람이 명확할 때만 넣어라.
- action_label은 한국어로 자연스럽게 짧게 써라.
- objects는 실제 사진 팝업 연결에 쓸 수 있는 명사 위주로 넣어라.
""".strip()


def gpt_scene_extract(row: pd.Series) -> List[Dict[str, Any]]:
    import requests

    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY 환경변수가 필요합니다.")

    payload = {
        "model": OPENAI_MODEL,
        "input": [
            {"role": "system", "content": GPT_SCENE_SYSTEM_PROMPT},
            {"role": "user", "content": build_gpt_scene_prompt(row)}
        ],
        "text": {
            "format": {
                "type": "json_schema",
                "name": "scene_extraction",
                "schema": {
                    "type": "object",
                    "properties": {
                        "scenes": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "summary": {"type": "string"},
                                    "canonical_action": {"type": "string"},
                                    "action_label": {"type": "string"},
                                    "category": {"type": "string"},
                                    "places": {"type": "array", "items": {"type": "string"}},
                                    "objects": {"type": "array", "items": {"type": "string"}},
                                    "companions": {"type": "array", "items": {"type": "string"}},
                                    "confidence": {"type": "number"}
                                },
                                "required": [
                                    "summary", "canonical_action", "action_label", "category",
                                    "places", "objects", "companions", "confidence"
                                ],
                                "additionalProperties": False
                            }
                        },
                        "noise_expressions": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": ["scenes", "noise_expressions"],
                    "additionalProperties": False
                }
            }
        }
    }

    resp = requests.post(
        "https://api.openai.com/v1/responses",
        headers={
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=120,
    )
    resp.raise_for_status()
    parsed = extract_json_from_text(resp.json()["output"][0]["content"][0]["text"])

    rule_hints = extract_rule_hints(row["content"])
    scenes = []

    for item in parsed.get("scenes", []):
        gpt_places = [str(v).strip() for v in item.get("places", []) if str(v).strip()]
        gpt_objects = [str(v).strip() for v in item.get("objects", []) if str(v).strip()]
        gpt_companions = [str(v).strip() for v in item.get("companions", []) if str(v).strip()]

        merged_places = merge_unique_preserve_order(gpt_places, rule_hints["places"])
        merged_objects = merge_unique_preserve_order(gpt_objects, rule_hints["objects"])
        merged_companions = merge_unique_preserve_order(gpt_companions, rule_hints["companions"])

        action_label = item.get("action_label", "").strip() or "한줄 기록"
        free_keywords = merge_unique_preserve_order(
            extract_free_keywords(item.get("summary", "")),
            rule_hints["free_keywords"],
        )
        keyword_bundle = build_keyword_bundle(
            action_label,
            merged_places,
            merged_objects,
            merged_companions,
            free_keywords,
        )

        scenes.append({
            "user_id": row["user_id"],
            "diary_id": row["diary_id"],
            "date": row["date"],
            "month": row["month"],
            "created_at": row["created_at"],
            "raw_text": item.get("summary", row["content"]),
            "content": row["content"],
            "canonical_action": item.get("canonical_action", "one_line_scene"),
            "action_label": action_label,
            "category": item.get("category", "기타"),
            "places": merged_places,
            "objects": merged_objects,
            "companions": merged_companions,
            "confidence": float(item.get("confidence", 0.7)),
            "source_image_url": row["image_url"],
            "source_full_image_url": row["full_image_url"],
            "photo_keywords": keyword_bundle["photo_keywords"],
            "photo_keyword_types": keyword_bundle["photo_keyword_types"],
        })

    if not scenes:
        scenes = rule_based_scene_extract(row)

    return scenes


all_scenes = []
for _, row in df.iterrows():
    if USE_GPT:
        try:
            scenes = gpt_scene_extract(row)
        except Exception as e:
            print(f"[GPT scene 추출 실패 -> 규칙 기반 fallback] diary_id={row['diary_id']}: {e}")
            scenes = rule_based_scene_extract(row)
    else:
        scenes = rule_based_scene_extract(row)
    all_scenes.extend(scenes)

scenes_df = pd.DataFrame(all_scenes)
scenes_df["date"] = pd.to_datetime(scenes_df["date"])
scenes_df["month"] = scenes_df["date"].dt.to_period("M").astype(str)


def normalize_list(x):
    if isinstance(x, list):
        return sorted(set([str(v).strip() for v in x if str(v).strip()]))
    return []


for col in ["places", "objects", "companions", "photo_keywords"]:
    scenes_df[col] = scenes_df[col].apply(normalize_list)

print("총 scene 수:", len(scenes_df))
display(scenes_df.head(10))



/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


총 scene 수: 659


,user_id,diary_id,date,month,created_at,raw_text,content,canonical_action,action_label,category,places,objects,companions,confidence,source_image_url,source_full_image_url,photo_keywords,photo_keyword_types
0,4,16,2025-12-12,2025-12,2025-12-12T15:56:10.957176,차 안에서 간식을 먹고 있음,따뜻한 차 안에서 간식 먹는 행복 💕🍪🚗,eat_snack_in_car,차 안에서 간식 먹기,식사/간식,[차 안],[간식],[],0.90,/media/images/20251212_155610_740766d2.jpg,http://3.35.185.251:8000/media/images/20251212...,"[간식, 간식을, 먹고, 안에서, 있음, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차 안': 'place', '간식'..."
1,4,17,2025-12-12,2025-12,2025-12-12T15:56:39.146318,장난감을 손에 들고 상상 놀이를 함,장난감 손에 들고 상상의 모험 떠나요🌟🔫🧙‍♂️,holding_toy_and_imagining_adventure,장난감 들고 상상 놀이,역할놀이,[],[장난감],[],0.90,/media/images/20251212_155639_e5c6a750.jpg,http://3.35.185.251:8000/media/images/20251212...,"[놀이를, 들고, 상상, 손에, 장난감, 장난감 들고 상상 놀이, 장난감을]","{'장난감 들고 상상 놀이': 'action', '장난감': 'object', '놀..."
2,4,18,2025-12-12,2025-12,2025-12-12T15:57:11.014722,고기 냄새를 맡으며 식사하는 어린이,고기 냄새에 반한 작은 미식가의 행복한 한 끼🍖💖,smelling_and_eating_meat,고기 냄새 맡으며 식사,식사/간식,[],[고기],[],0.90,/media/images/20251212_155711_d6367e62.jpg,http://3.35.185.251:8000/media/images/20251212...,"[고기, 고기 냄새 맡으며 식사, 냄새를, 맡으며, 식사하는, 어린이]","{'고기 냄새 맡으며 식사': 'action', '고기': 'object', '냄새..."
3,4,19,2025-12-12,2025-12,2025-12-12T15:57:31.413000,고기를 굽는 장면,따뜻한 고기 굽는 불빛에 마음이 포근해져요🔥🥩❤️,grilling_meat,고기 굽기,식사/간식,[],"[고기, 불빛]",[],0.95,/media/images/20251212_155731_d5e9e5ee.jpg,http://3.35.185.251:8000/media/images/20251212...,"[고기, 고기 굽기, 고기를, 굽는, 마음이, 불빛, 장면]","{'고기 굽기': 'action', '고기': 'object', '불빛': 'obj..."
4,4,20,2025-12-12,2025-12,2025-12-12T15:57:51.405639,불꽃 앞에서 손을 잡음,따뜻한 불꽃 아래서 손 꼭 잡았어요🔥🤝✨,holding_hands_near_fire,손 잡기,가족상호작용,[],[불꽃],[],0.90,/media/images/20251212_155751_43641256.jpg,http://3.35.185.251:8000/media/images/20251212...,"[불꽃, 손 잡기, 손을, 아래서, 앞에서, 잡음]","{'손 잡기': 'action', '불꽃': 'object', '손을': 'free..."
5,4,21,2025-12-12,2025-12,2025-12-12T17:37:22.897148,차 안에서 간식을 먹는 모습,"차 안에서 간식 타임, 행복 가득🍪🚗💖",eating_snack_in_car,차 안에서 간식 먹기,식사/간식,[차 안],[간식],[],1.00,/media/images/20251212_173722_ea92a36b.jpg,http://3.35.185.251:8000/media/images/20251212...,"[간식, 간식을, 먹는, 모습, 안에서, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차 안': 'place', '간식'..."
6,4,22,2025-12-12,2025-12,2025-12-12T17:53:42.964589,차 안에서 간식을 먹는 장면,차 안에서 따뜻한 간식 먹으며 행복한 시간🍪🚗💖,eating_snack_in_car,차 안에서 간식 먹기,식사/간식,"[차, 차 안]",[간식],[],1.00,/media/images/20251212_175342_69a20169.jpg,http://3.35.185.251:8000/media/images/20251212...,"[간식, 간식을, 먹는, 안에서, 장면, 차, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차': 'place', '차 안':..."
7,4,23,2025-12-12,2025-12,2025-12-12T18:04:14.040147,작은 손에 장난감 권총을 들고 있다,작은 손에 큰 모험이 시작되었어요🔫🕵️‍♂️✨,holding_toy_gun,장난감 권총 들기,놀이,[],[장난감 권총],[],0.90,/media/images/20251212_180414_940ae3ae.jpg,http://3.35.185.251:8000/media/images/20251212...,"[권총을, 들고, 손에, 있다, 장난감, 장난감 권총, 장난감 권총 들기]","{'장난감 권총 들기': 'action', '장난감 권총': 'object', '권..."
8,4,24,2025-12-12,2025-12,2025-12-12T18:18:07.162682,따뜻한 불꽃 주변에서 손을 잡고 있다,따뜻한 불꽃 속에서 손 꼭 잡고 🌟🔥💖,holding_hands_near_warm_fire,손잡기,가족상호작용,[],[불꽃],[],0.90,/media/images/20251212_181807_449a10c0.jpg,http://3.35.185.251:8000/media/images/20251212...,"[불꽃, 손을, 손잡기, 있다, 잡고, 주변에서]","{'손잡기': 'action', '불꽃': 'object', '손을': 'free'..."
9,4,26,2025-12-13,2025-12,2025-12-13T01:15:09.831776,차 안에서 간식을 먹음,따끈한 간식과 함께 달콤한 차 안 시간🍪🚗💨,eating_snack_in_car,차 안에서 간식 먹기,식사/간식,[차 안],"[간식, 차]",[],1.00,/media/images/20251213_011509_8bb7b6cd.jpg,http://3.35.185.251:8000/media/images/20251213...,"[간식, 간식과, 간식을, 따끈한, 먹음, 안에서, 차, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차 안': 'place', '간식'..."


In [11]:
import json
import pandas as pd

SCENE_CACHE_PATH = "./scene_extraction_results_v3.json"

with open(SCENE_CACHE_PATH, "r", encoding="utf-8") as f:
    cached_scenes = json.load(f)

scenes_df = pd.DataFrame(cached_scenes)
scenes_df["date"] = pd.to_datetime(scenes_df["date"])
scenes_df["month"] = scenes_df["date"].dt.to_period("M").astype(str)

def normalize_list(x):
    if isinstance(x, list):
        return sorted(set([str(v).strip() for v in x if str(v).strip()]))
    return []

for col in ["places", "objects", "companions", "photo_keywords"]:
    if col in scenes_df.columns:
        scenes_df[col] = scenes_df[col].apply(normalize_list)

print("cached scene 수:", len(scenes_df))
display(scenes_df.head())


cached scene 수: 659


,user_id,diary_id,date,month,created_at,raw_text,content,canonical_action,action_label,category,places,objects,companions,confidence,source_image_url,source_full_image_url,photo_keywords,photo_keyword_types
0,4,16,2025-12-12,2025-12,2025-12-12T15:56:10.957176,차 안에서 간식을 먹고 있음,따뜻한 차 안에서 간식 먹는 행복 💕🍪🚗,eat_snack_in_car,차 안에서 간식 먹기,식사/간식,[차 안],[간식],[],0.90,/media/images/20251212_155610_740766d2.jpg,http://3.35.185.251:8000/media/images/20251212...,"[간식, 간식을, 먹고, 안에서, 있음, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차 안': 'place', '간식'..."
1,4,17,2025-12-12,2025-12,2025-12-12T15:56:39.146318,장난감을 손에 들고 상상 놀이를 함,장난감 손에 들고 상상의 모험 떠나요🌟🔫🧙‍♂️,holding_toy_and_imagining_adventure,장난감 들고 상상 놀이,역할놀이,[],[장난감],[],0.90,/media/images/20251212_155639_e5c6a750.jpg,http://3.35.185.251:8000/media/images/20251212...,"[놀이를, 들고, 상상, 손에, 장난감, 장난감 들고 상상 놀이, 장난감을]","{'장난감 들고 상상 놀이': 'action', '장난감': 'object', '놀..."
2,4,18,2025-12-12,2025-12,2025-12-12T15:57:11.014722,고기 냄새를 맡으며 식사하는 어린이,고기 냄새에 반한 작은 미식가의 행복한 한 끼🍖💖,smelling_and_eating_meat,고기 냄새 맡으며 식사,식사/간식,[],[고기],[],0.90,/media/images/20251212_155711_d6367e62.jpg,http://3.35.185.251:8000/media/images/20251212...,"[고기, 고기 냄새 맡으며 식사, 냄새를, 맡으며, 식사하는, 어린이]","{'고기 냄새 맡으며 식사': 'action', '고기': 'object', '냄새..."
3,4,19,2025-12-12,2025-12,2025-12-12T15:57:31.413000,고기를 굽는 장면,따뜻한 고기 굽는 불빛에 마음이 포근해져요🔥🥩❤️,grilling_meat,고기 굽기,식사/간식,[],"[고기, 불빛]",[],0.95,/media/images/20251212_155731_d5e9e5ee.jpg,http://3.35.185.251:8000/media/images/20251212...,"[고기, 고기 굽기, 고기를, 굽는, 마음이, 불빛, 장면]","{'고기 굽기': 'action', '고기': 'object', '불빛': 'obj..."
4,4,20,2025-12-12,2025-12,2025-12-12T15:57:51.405639,불꽃 앞에서 손을 잡음,따뜻한 불꽃 아래서 손 꼭 잡았어요🔥🤝✨,holding_hands_near_fire,손 잡기,가족상호작용,[],[불꽃],[],0.90,/media/images/20251212_155751_43641256.jpg,http://3.35.185.251:8000/media/images/20251212...,"[불꽃, 손 잡기, 손을, 아래서, 앞에서, 잡음]","{'손 잡기': 'action', '불꽃': 'object', '손을': 'free..."


In [ ]:
if "OPENAI_API_KEY" not in globals():
    OPENAI_API_KEY = YOUR_API_KEY
if "OPENAI_MODEL" not in globals():
    OPENAI_MODEL = "gpt-4.1-mini"
if "extract_json_from_text" not in globals():
    def extract_json_from_text(text: str) -> Dict[str, Any]:
        text = str(text).strip()
        try:
            return json.loads(text)
        except Exception:
            pass

        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))

        raise ValueError("JSON 파싱 실패")

USE_LLM_SYNONYM_GROUPING = True
SYNONYM_MODEL = OPENAI_MODEL


def flatten_counter(series_of_lists: pd.Series) -> Counter:
    c = Counter()
    for items in series_of_lists:
        if isinstance(items, list):
            c.update(items)
    return c


def dedupe_photo_records(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for item in records:
        key = (item.get("diary_id"), item.get("full_image_url"))
        if key in seen:
            continue
        seen.add(key)
        out.append(item)
    return out


def build_identity_synonym_map(items: List[str]) -> Dict[str, str]:
    return {item: item for item in items}


def build_synonym_groups_with_llm(user_id: int, month: str, items: List[str], label: str) -> Dict[str, Any]:
    import requests

    items = [str(item).strip() for item in items if str(item).strip()]
    items = sorted(dict.fromkeys(items))
    if len(items) <= 1:
        return {
            "synonym_map": build_identity_synonym_map(items),
            "groups": [{"canonical": item, "members": [item]} for item in items]
        }

    if not USE_LLM_SYNONYM_GROUPING or not OPENAI_API_KEY:
        return {
            "synonym_map": build_identity_synonym_map(items),
            "groups": [{"canonical": item, "members": [item]} for item in items]
        }

    system_prompt = f"""
너는 육아 리포트 데이터에서 {label} 후보들이 서로 사실상 같은 의미인지 판정하는 정규화 도우미다.

규칙:
- 같은 의미거나 부모가 보기엔 사실상 같은 장면/키워드면 하나의 그룹으로 묶는다.
- 의미가 다르거나 사건의 느낌이 다르면 절대 합치지 않는다.
- 공격적으로 합치지 말고 보수적으로 판단한다.
- canonical은 가장 자연스럽고 대표적인 한국어 표현으로 정한다.
- 반드시 JSON만 출력한다.

출력 형식:
{{
  "groups": [
    {{"canonical": "대표어", "members": ["원어1", "원어2"]}}
  ]
}}
""".strip()

    user_prompt = json.dumps({
        "user_id": user_id,
        "month": month,
        "label": label,
        "items": items,
    }, ensure_ascii=False, indent=2)

    payload = {
        "model": SYNONYM_MODEL,
        "input": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "text": {
            "format": {
                "type": "json_schema",
                "name": "synonym_groups",
                "schema": {
                    "type": "object",
                    "properties": {
                        "groups": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "canonical": {"type": "string"},
                                    "members": {"type": "array", "items": {"type": "string"}}
                                },
                                "required": ["canonical", "members"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["groups"],
                    "additionalProperties": False
                }
            }
        }
    }

    try:
        resp = requests.post(
            "https://api.openai.com/v1/responses",
            headers={
                "Authorization": f"Bearer {OPENAI_API_KEY}",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=120,
        )
        resp.raise_for_status()
        parsed = extract_json_from_text(resp.json()["output"][0]["content"][0]["text"])
    except Exception as e:
        print(f"[동의어 군집화 실패 -> identity fallback] user={user_id} month={month} label={label}: {e}")
        return {
            "synonym_map": build_identity_synonym_map(items),
            "groups": [{"canonical": item, "members": [item]} for item in items]
        }

    synonym_map = {}
    groups = []
    seen = set()
    for group in parsed.get("groups", []):
        canonical = str(group.get("canonical", "")).strip()
        members = [str(x).strip() for x in group.get("members", []) if str(x).strip()]
        members = [m for m in members if m in items]
        if not members:
            continue
        if not canonical:
            canonical = members[0]
        groups.append({
            "canonical": canonical,
            "members": members,
        })
        for member in members:
            synonym_map[member] = canonical
            seen.add(member)

    for item in items:
        if item not in seen:
            synonym_map[item] = item
            groups.append({"canonical": item, "members": [item]})

    return {
        "synonym_map": synonym_map,
        "groups": groups
    }


def build_keyword_photo_index(mdf: pd.DataFrame, keyword_synonym_map: Dict[str, str]) -> Dict[str, Any]:
    keyword_map = {}
    for _, row in mdf.iterrows():
        for keyword in row["photo_keywords"]:
            canonical_keyword = keyword_synonym_map.get(keyword, keyword)
            bucket = keyword_map.setdefault(canonical_keyword, {
                "keyword": canonical_keyword,
                "keyword_type": row["photo_keyword_types"].get(keyword, "free"),
                "aliases": set(),
                "dates": set(),
                "photos": []
            })
            current_type = row["photo_keyword_types"].get(keyword, "free")
            if bucket["keyword_type"] == "free" and current_type != "free":
                bucket["keyword_type"] = current_type
            bucket["aliases"].add(keyword)
            bucket["dates"].add(str(row["date"].date()))
            bucket["photos"].append({
                "diary_id": row["diary_id"],
                "date": str(row["date"].date()),
                "image_url": row["source_image_url"],
                "full_image_url": row["source_full_image_url"],
                "content": row["content"]
            })

    final_index = {}
    for canonical_keyword, payload in keyword_map.items():
        photos = dedupe_photo_records(payload["photos"])
        final_index[canonical_keyword] = {
            "keyword": canonical_keyword,
            "keyword_type": payload["keyword_type"],
            "aliases": sorted(payload["aliases"]),
            "dates": sorted(payload["dates"]),
            "photo_count": len(photos),
            "photos": photos
        }
    return final_index


def compute_monthly_stats_by_user(sdf: pd.DataFrame) -> Dict[int, Dict[str, Any]]:
    result = {}
    for user_id, user_df in sdf.groupby("user_id"):
        monthly = {}
        months = sorted(user_df["month"].unique())
        prev_semantic_actions = set()

        for month in months:
            mdf = user_df[user_df["month"] == month].copy()

            action_terms = sorted(set(mdf["action_label"].dropna().tolist()))
            action_grouping = build_synonym_groups_with_llm(user_id, month, action_terms, "행동 라벨")
            action_synonym_map = action_grouping["synonym_map"]
            mdf["semantic_action"] = mdf["action_label"].map(lambda x: action_synonym_map.get(x, x))

            raw_keyword_terms = sorted(set(flatten_counter(mdf["photo_keywords"]).keys()))
            keyword_grouping = build_synonym_groups_with_llm(user_id, month, raw_keyword_terms, "사진 키워드")
            keyword_synonym_map = keyword_grouping["synonym_map"]
            mdf["semantic_keywords"] = mdf["photo_keywords"].apply(
                lambda items: merge_unique_preserve_order([keyword_synonym_map.get(item, item) for item in items])
            )

            action_counter = Counter(mdf["semantic_action"].dropna().tolist())
            category_counter = Counter(mdf["category"].dropna().tolist())
            place_counter = flatten_counter(mdf["places"])
            object_counter = flatten_counter(mdf["objects"])
            companion_counter = flatten_counter(mdf["companions"])

            current_semantic_actions = set(action_counter.keys())
            new_actions = sorted(list(current_semantic_actions - prev_semantic_actions))
            prev_semantic_actions = current_semantic_actions

            monthly[month] = {
                "num_scenes": len(mdf),
                "top_actions": action_counter.most_common(5),
                "top_categories": category_counter.most_common(5),
                "top_places": place_counter.most_common(5),
                "top_objects": object_counter.most_common(5),
                "top_companions": companion_counter.most_common(5),
                "new_actions": new_actions,
                "action_synonym_map": action_synonym_map,
                "action_synonym_groups": action_grouping["groups"],
                "keyword_synonym_map": keyword_synonym_map,
                "keyword_synonym_groups": keyword_grouping["groups"],
                "keyword_photo_index": build_keyword_photo_index(mdf, keyword_synonym_map),
                "raw_df": mdf
            }

        result[int(user_id)] = monthly
    return result


monthly_stats_by_user = compute_monthly_stats_by_user(scenes_df)
sorted(monthly_stats_by_user.keys())




/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[4, 15, 16, 18, 19, 20]

In [13]:
def score_scene(row: pd.Series, month_stat: Dict[str, Any]) -> Dict[str, float]:
    top_actions = dict(month_stat["top_actions"])
    top_companions = dict(month_stat["top_companions"])
    new_actions = set(month_stat["new_actions"])

    semantic_action = row.get("semantic_action", row["action_label"])
    representativeness = top_actions.get(semantic_action, 0)
    novelty = 2.0 if semantic_action in new_actions else 0.0
    specificity = len(row["places"]) + len(row["companions"]) + len(row["objects"])
    family_signal = sum(top_companions.get(c, 0) for c in row["companions"])
    scene_confidence = float(row.get("confidence", 0.5))

    total = (
        1.5 * representativeness +
        2.0 * novelty +
        1.2 * specificity +
        0.8 * family_signal +
        1.0 * scene_confidence
    )

    return {
        "representativeness": representativeness,
        "novelty": novelty,
        "specificity": specificity,
        "family_signal": family_signal,
        "scene_confidence": scene_confidence,
        "total": total
    }


def select_highlights(month_stat: Dict[str, Any], top_k: int = 3) -> pd.DataFrame:
    mdf = month_stat["raw_df"].copy()
    scores = mdf.apply(lambda r: score_scene(r, month_stat), axis=1)
    score_df = pd.DataFrame(list(scores))
    out = pd.concat([mdf.reset_index(drop=True), score_df], axis=1)

    out = out.sort_values(
        by=["total", "specificity", "representativeness", "scene_confidence"],
        ascending=False
    ).copy()

    out["semantic_action"] = out["semantic_action"].fillna(out["action_label"])
    out["highlight_text"] = out.apply(
        lambda r: f"{str(r['date'].date())}: {r['action_label']} | 대표군={r['semantic_action']} | 장소={', '.join(r['places']) if r['places'] else '-'} | "
                  f"대상={', '.join(r['objects']) if r['objects'] else '-'} | 함께한 사람={', '.join(r['companions']) if r['companions'] else '-'}",
        axis=1
    )

    selected_indices = []
    seen_semantic_actions = set()
    for idx, row in out.iterrows():
        semantic_action = row.get("semantic_action", row["action_label"])
        if semantic_action in seen_semantic_actions:
            continue
        selected_indices.append(idx)
        seen_semantic_actions.add(semantic_action)
        if len(selected_indices) >= top_k:
            break

    if len(selected_indices) < top_k:
        for idx in out.index:
            if idx in selected_indices:
                continue
            selected_indices.append(idx)
            if len(selected_indices) >= top_k:
                break

    return out.loc[selected_indices].reset_index(drop=True)


monthly_highlights_by_user = {
    user_id: {
        month: select_highlights(stat, top_k=3)
        for month, stat in monthly.items()
    }
    for user_id, monthly in monthly_stats_by_user.items()
}

list(monthly_highlights_by_user.keys())



[4, 15, 16, 18, 19, 20]

In [14]:
USE_GPT_REPORT = True

REPORT_SYSTEM_PROMPT = """
너는 '부모에게 전달되는 월간 육아 리포트'를 작성하는 전문가다.

목표:
- 데이터 기반이지만, 따뜻하고 자연스러운 문장으로 작성한다.
- 부모가 읽었을 때 아이의 한 달이 떠오르도록 쓴다.

작성 스타일:
- 과장된 감성 표현은 금지
- 담백하고 따뜻하게
- 관찰 기반 + 해석 형태로 작성

출력 형식 (JSON):
{
  "month_overview": "...",
  "pattern_summary": "...",
  "change_summary": "...",
  "parent_note": "...",
  "one_line_summary": "..."
}
""".strip()


def build_month_report_prompt(user_id: int, month: str, stat: Dict[str, Any], highlight_df: pd.DataFrame) -> str:
    payload = {
        "user_id": user_id,
        "month": month,
        "top_actions": stat["top_actions"],
        "top_categories": stat["top_categories"],
        "top_places": stat["top_places"],
        "top_objects": stat["top_objects"],
        "top_companions": stat["top_companions"],
        "new_actions": stat["new_actions"],
        "action_synonym_groups": stat.get("action_synonym_groups", []),
        "keyword_synonym_groups": stat.get("keyword_synonym_groups", []),
        "highlights": highlight_df[[
            "date", "action_label", "semantic_action", "places", "objects", "companions",
            "raw_text", "content", "photo_keywords", "semantic_keywords", "total"
        ]].copy().astype(str).to_dict(orient="records")
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def gpt_compose_month_report(user_id: int, month: str, stat: Dict[str, Any], highlight_df: pd.DataFrame) -> Dict[str, Any]:
    import requests

    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY 환경변수가 필요합니다.")

    url = "https://api.openai.com/v1/responses"
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": OPENAI_MODEL,
        "input": [
            {"role": "system", "content": REPORT_SYSTEM_PROMPT},
            {"role": "user", "content": build_month_report_prompt(user_id, month, stat, highlight_df)}
        ],
        "text": {
            "format": {
                "type": "json_schema",
                "name": "monthly_report",
                "schema": {
                    "type": "object",
                    "properties": {
                        "month_overview": {"type": "string"},
                        "pattern_summary": {"type": "string"},
                        "change_summary": {"type": "string"},
                        "parent_note": {"type": "string"},
                        "one_line_summary": {"type": "string"}
                    },
                    "required": [
                        "month_overview", "pattern_summary", "change_summary",
                        "parent_note", "one_line_summary"
                    ],
                    "additionalProperties": False
                }
            }
        }
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    text_output = data["output"][0]["content"][0]["text"]
    return json.loads(text_output)


def annotate_text(text: str, keyword_photo_index: Dict[str, Any]) -> List[Dict[str, Any]]:
    annotations = []
    for canonical_keyword, payload in keyword_photo_index.items():
        candidates = [canonical_keyword] + payload.get("aliases", [])
        seen_alias = set()
        for candidate in candidates:
            candidate = str(candidate).strip()
            if not candidate or candidate in seen_alias:
                continue
            seen_alias.add(candidate)
            start = 0
            while True:
                idx = text.find(candidate, start)
                if idx == -1:
                    break
                annotations.append({
                    "start": idx,
                    "end": idx + len(candidate),
                    "keyword": canonical_keyword,
                    "matched_text": candidate,
                    "keyword_type": payload["keyword_type"],
                    "dates": payload["dates"],
                    "photo_count": payload["photo_count"],
                    "photos": payload["photos"],
                    "aliases": payload.get("aliases", [])
                })
                start = idx + len(candidate)
    annotations.sort(key=lambda item: (item["start"], -(item["end"] - item["start"])))
    return annotations


def make_rule_based_month_report(month: str, stat: Dict[str, Any], highlight_df: pd.DataFrame) -> Dict[str, Any]:
    top_actions = [x[0] for x in stat["top_actions"][:3]]
    top_places = [x[0] for x in stat["top_places"][:3]]
    top_objects = [x[0] for x in stat["top_objects"][:3]]
    top_companions = [x[0] for x in stat["top_companions"][:3]]
    new_actions = stat["new_actions"][:3]

    month_overview = (
        f"{month}에는 {', '.join(top_actions)} 활동이 반복적으로 기록되었다."
        if top_actions else f"{month}에는 다양한 장면이 기록되었다."
    )
    pattern_summary = " ".join([
        f"주요 공간은 {', '.join(top_places)}였다." if top_places else "",
        f"자주 보인 대상은 {', '.join(top_objects)}였다." if top_objects else "",
        f"함께한 인물로는 {', '.join(top_companions)}이(가) 자주 기록되었다." if top_companions else ""
    ]).strip()
    change_summary = (
        f"새롭게 두드러진 활동으로는 {', '.join(new_actions)}가 있었다."
        if new_actions else "전달과 이어지는 활동 흐름이 유지되었다."
    )
    parent_note = "리포트 안의 키워드를 누르면 해당 장면과 연결된 사진을 다시 볼 수 있게 구성했다."
    one_line_summary = (
        f"{month}은(는) {', '.join(top_actions[:2])} 중심으로 장면이 쌓인 달이었다."
        if top_actions else f"{month}은(는) 사진과 한줄일기가 차곡차곡 쌓인 달이었다."
    )

    report = {
        "month": month,
        "mode": "rule",
        "month_overview": month_overview,
        "pattern_summary": pattern_summary,
        "change_summary": change_summary,
        "parent_note": parent_note,
        "one_line_summary": one_line_summary,
        "highlights": []
    }

    for _, row in highlight_df.iterrows():
        highlight_text = (
            f"{str(row['date'].date())}에는 {row['action_label']} 장면이 눈에 띄었다. "
            f"대표적으로는 {row.get('semantic_action', row['action_label'])} 흐름으로 묶이는 장면이었다. "
            f"장소는 {', '.join(row['places']) if row['places'] else '미상'}였고, "
            f"대상은 {', '.join(row['objects']) if row['objects'] else '특정 물건 없음'}이었다."
        )

        date_mask = stat["raw_df"]["date"].dt.date == row["date"].date()
        date_rows = stat["raw_df"][date_mask]
        date_photos = dedupe_photo_records([
            {
                "diary_id": r["diary_id"],
                "date": str(r["date"].date()),
                "image_url": r["source_image_url"],
                "full_image_url": r["source_full_image_url"],
                "content": r["content"]
            }
            for _, r in date_rows.iterrows()
        ])

        keywords = []
        for keyword in [row.get('semantic_action', row['action_label']), row['action_label'], *row['places'], *row['objects'], *row['companions']]:
            if keyword and keyword in stat["keyword_photo_index"]:
                keywords.append(stat["keyword_photo_index"][keyword])

        dedup_keywords = {}
        for payload in keywords:
            dedup_keywords[payload['keyword']] = payload

        report["highlights"].append({
            "date": str(row["date"].date()),
            "text": highlight_text,
            "raw_text": row["raw_text"],
            "action_label": row["action_label"],
            "semantic_action": row.get("semantic_action", row["action_label"]),
            "source_diary_id": row["diary_id"],
            "source_image_url": row["source_image_url"],
            "source_full_image_url": row["source_full_image_url"],
            "keywords": list(dedup_keywords.values()),
            "annotations": annotate_text(highlight_text, {k: v for k, v in dedup_keywords.items()}),
            "fallback_photos": date_photos
        })

    report["keyword_annotations"] = {
        field: annotate_text(report[field], stat["keyword_photo_index"])
        for field in ["month_overview", "pattern_summary", "change_summary", "parent_note", "one_line_summary"]
    }
    report["keyword_photo_index"] = stat["keyword_photo_index"]
    report["action_synonym_groups"] = stat.get("action_synonym_groups", [])
    report["keyword_synonym_groups"] = stat.get("keyword_synonym_groups", [])
    report["photo_library"] = dedupe_photo_records([
        {
            "diary_id": r["diary_id"],
            "date": str(r["date"].date()),
            "image_url": r["source_image_url"],
            "full_image_url": r["source_full_image_url"],
            "content": r["content"]
        }
        for _, r in stat["raw_df"].iterrows()
    ])
    return report



In [15]:
final_month_reports = {}

for user_id, monthly in monthly_stats_by_user.items():
    final_month_reports[user_id] = {}
    for month in sorted(monthly.keys()):
        stat = monthly[month]
        hi = monthly_highlights_by_user[user_id][month].copy()
        if "date" in hi.columns:
            hi["date"] = pd.to_datetime(hi["date"])

        if USE_GPT_REPORT:
            try:
                gpt_report = gpt_compose_month_report(user_id, month, stat, hi)
                report = {
                    "month": month,
                    "mode": "gpt",
                    **gpt_report,
                    "highlights": make_rule_based_month_report(month, stat, hi)["highlights"],
                    "keyword_annotations": {
                        field: annotate_text(gpt_report[field], stat["keyword_photo_index"])
                        for field in ["month_overview", "pattern_summary", "change_summary", "parent_note", "one_line_summary"]
                    },
                    "keyword_photo_index": stat["keyword_photo_index"],
                    "photo_library": make_rule_based_month_report(month, stat, hi)["photo_library"]
                }
            except Exception as e:
                print(f"[GPT 리포트 실패 -> 규칙 기반 fallback] user={user_id} month={month}: {e}")
                report = make_rule_based_month_report(month, stat, hi)
        else:
            report = make_rule_based_month_report(month, stat, hi)

        final_month_reports[user_id][month] = report

list(final_month_reports.keys())


[4, 15, 16, 18, 19, 20]

In [10]:
def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]
    elif isinstance(obj, pd.Timestamp):
        return str(obj)
    elif isinstance(obj, pd.Period):
        return str(obj)
    else:
        return obj


scene_save_path = "./scene_extraction_results_v3.json"
report_save_path = "./monthly_reports_v3.json"

scene_export = scenes_df.copy()
scene_export["date"] = scene_export["date"].astype(str)

with open(scene_save_path, "w", encoding="utf-8") as f:
    json.dump(scene_export.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

with open(report_save_path, "w", encoding="utf-8") as f:
    json.dump(make_json_serializable(final_month_reports), f, ensure_ascii=False, indent=2)

print("저장 준비 완료")
print(scene_save_path)
print(report_save_path)


저장 준비 완료
./scene_extraction_results_v3.json
./monthly_reports_v3.json


In [16]:
user_to_inspect = sorted(monthly_stats_by_user.keys())[0]
month_to_inspect = sorted(monthly_stats_by_user[user_to_inspect].keys())[0]

print("선택 사용자:", user_to_inspect)
print("선택 월:", month_to_inspect)
display(monthly_stats_by_user[user_to_inspect][month_to_inspect]["raw_df"].head(20))
display(monthly_highlights_by_user[user_to_inspect][month_to_inspect][[
    "date", "action_label", "category", "places", "objects", "companions",
    "photo_keywords", "source_image_url", "representativeness", "novelty",
    "specificity", "total", "raw_text"
]])
display(final_month_reports[user_to_inspect][month_to_inspect])


선택 사용자: 4
선택 월: 2025-12


,user_id,diary_id,date,month,created_at,raw_text,content,canonical_action,action_label,category,places,objects,companions,confidence,source_image_url,source_full_image_url,photo_keywords,photo_keyword_types,semantic_action,semantic_keywords
0,4,16,2025-12-12,2025-12,2025-12-12T15:56:10.957176,차 안에서 간식을 먹고 있음,따뜻한 차 안에서 간식 먹는 행복 💕🍪🚗,eat_snack_in_car,차 안에서 간식 먹기,식사/간식,[차 안],[간식],[],0.90,/media/images/20251212_155610_740766d2.jpg,http://3.35.185.251:8000/media/images/20251212...,"[간식, 간식을, 먹고, 안에서, 있음, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차 안': 'place', '간식'...",차 안에서 간식 먹기,"[간식, 먹는 모습, 안에서 있음, 차]"
1,4,17,2025-12-12,2025-12,2025-12-12T15:56:39.146318,장난감을 손에 들고 상상 놀이를 함,장난감 손에 들고 상상의 모험 떠나요🌟🔫🧙‍♂️,holding_toy_and_imagining_adventure,장난감 들고 상상 놀이,역할놀이,[],[장난감],[],0.90,/media/images/20251212_155639_e5c6a750.jpg,http://3.35.185.251:8000/media/images/20251212...,"[놀이를, 들고, 상상, 손에, 장난감, 장난감 들고 상상 놀이, 장난감을]","{'장난감 들고 상상 놀이': 'action', '장난감': 'object', '놀...",장난감 들고 상상 놀이,"[놀이, 들고 있음, 상상, 장난감, 권총]"
2,4,18,2025-12-12,2025-12,2025-12-12T15:57:11.014722,고기 냄새를 맡으며 식사하는 어린이,고기 냄새에 반한 작은 미식가의 행복한 한 끼🍖💖,smelling_and_eating_meat,고기 냄새 맡으며 식사,식사/간식,[],[고기],[],0.90,/media/images/20251212_155711_d6367e62.jpg,http://3.35.185.251:8000/media/images/20251212...,"[고기, 고기 냄새 맡으며 식사, 냄새를, 맡으며, 식사하는, 어린이]","{'고기 냄새 맡으며 식사': 'action', '고기': 'object', '냄새...",고기 냄새 맡으며 식사,"[고기, 고기 냄새 맡으며 식사, 먹는 모습, 아이]"
3,4,19,2025-12-12,2025-12,2025-12-12T15:57:31.413000,고기를 굽는 장면,따뜻한 고기 굽는 불빛에 마음이 포근해져요🔥🥩❤️,grilling_meat,고기 굽기,식사/간식,[],"[고기, 불빛]",[],0.95,/media/images/20251212_155731_d5e9e5ee.jpg,http://3.35.185.251:8000/media/images/20251212...,"[고기, 고기 굽기, 고기를, 굽는, 마음이, 불빛, 장면]","{'고기 굽기': 'action', '고기': 'object', '불빛': 'obj...",고기 굽기,"[고기, 고기 굽기, 마음이, 불빛, 모습]"
4,4,20,2025-12-12,2025-12,2025-12-12T15:57:51.405639,불꽃 앞에서 손을 잡음,따뜻한 불꽃 아래서 손 꼭 잡았어요🔥🤝✨,holding_hands_near_fire,손 잡기,가족상호작용,[],[불꽃],[],0.90,/media/images/20251212_155751_43641256.jpg,http://3.35.185.251:8000/media/images/20251212...,"[불꽃, 손 잡기, 손을, 아래서, 앞에서, 잡음]","{'손 잡기': 'action', '불꽃': 'object', '손을': 'free...",손 잡기,"[불빛, 들고 있음, 아래에서, 앞에서]"
5,4,21,2025-12-12,2025-12,2025-12-12T17:37:22.897148,차 안에서 간식을 먹는 모습,"차 안에서 간식 타임, 행복 가득🍪🚗💖",eating_snack_in_car,차 안에서 간식 먹기,식사/간식,[차 안],[간식],[],1.00,/media/images/20251212_173722_ea92a36b.jpg,http://3.35.185.251:8000/media/images/20251212...,"[간식, 간식을, 먹는, 모습, 안에서, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차 안': 'place', '간식'...",차 안에서 간식 먹기,"[간식, 먹는 모습, 모습, 안에서 있음, 차]"
6,4,22,2025-12-12,2025-12,2025-12-12T17:53:42.964589,차 안에서 간식을 먹는 장면,차 안에서 따뜻한 간식 먹으며 행복한 시간🍪🚗💖,eating_snack_in_car,차 안에서 간식 먹기,식사/간식,"[차, 차 안]",[간식],[],1.00,/media/images/20251212_175342_69a20169.jpg,http://3.35.185.251:8000/media/images/20251212...,"[간식, 간식을, 먹는, 안에서, 장면, 차, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차': 'place', '차 안':...",차 안에서 간식 먹기,"[간식, 먹는 모습, 안에서 있음, 모습, 차]"
7,4,23,2025-12-12,2025-12,2025-12-12T18:04:14.040147,작은 손에 장난감 권총을 들고 있다,작은 손에 큰 모험이 시작되었어요🔫🕵️‍♂️✨,holding_toy_gun,장난감 권총 들기,놀이,[],[장난감 권총],[],0.90,/media/images/20251212_180414_940ae3ae.jpg,http://3.35.185.251:8000/media/images/20251212...,"[권총을, 들고, 손에, 있다, 장난감, 장난감 권총, 장난감 권총 들기]","{'장난감 권총 들기': 'action', '장난감 권총': 'object', '권...",장난감 권총 들기,"[권총, 들고 있음, 안에서 있음, 장난감]"
8,4,24,2025-12-12,2025-12,2025-12-12T18:18:07.162682,따뜻한 불꽃 주변에서 손을 잡고 있다,따뜻한 불꽃 속에서 손 꼭 잡고 🌟🔥💖,holding_hands_near_warm_fire,손잡기,가족상호작용,[],[불꽃],[],0.90,/media/images/20251212_181807_449a10c0.jpg,http://3.35.185.251:8000/media/images/20251212...,"[불꽃, 손을, 손잡기, 있다, 잡고, 주변에서]","{'손잡기': 'action', '불꽃': 'object', '손을': 'free'...",손 잡기,"[불빛, 들고 있음, 안에서 있음, 주변에서]"
9,4,26,2025-12-13,2025-12,2025-12-13T01:15:09.831776,차 안에서 간식을 먹음,따끈한 간식과 함께 달콤한 차 안 시간🍪🚗💨,eating_snack_in_car,차 안에서 간식 먹기,식사/간식,[차 안],"[간식, 차]",[],1.00,/media/images/20251213_011509_8bb7b6cd.jpg,http://3.35.185.251:8000/media/images/20251213...,"[간식, 간식과, 간식을, 따끈한, 먹음, 안에서, 차, 차 안, 차 안에서 간식 먹기]","{'차 안에서 간식 먹기': 'action', '차 안': 'place', '간식'...",차 안에서 간식 먹기,"[간식, 따끈한, 먹는 모습, 안에서 있음, 차]"

,date,action_label,category,places,objects,companions,photo_keywords,source_image_url,representativeness,novelty,specificity,total,raw_text
0,2025-12-12,차 안에서 간식 먹기,식사/간식,"[차, 차 안]",[간식],[],"[간식, 간식을, 먹는, 안에서, 장면, 차, 차 안, 차 안에서 간식 먹기]",/media/images/20251212_175342_69a20169.jpg,5,2.0,3,16.10,차 안에서 간식을 먹는 장면
1,2025-12-13,곰인형 안기,놀이,[],"[곰인형, 인형]",[],"[곰인형, 곰인형 안기, 곰인형을, 모습, 안고, 인형, 있는]",/media/images/20251213_025154_918ec38c.jpg,4,2.0,2,13.40,곰인형을 안고 있는 모습
2,2025-12-12,고기 굽기,식사/간식,[],"[고기, 불빛]",[],"[고기, 고기 굽기, 고기를, 굽는, 마음이, 불빛, 장면]",/media/images/20251212_155731_d5e9e5ee.jpg,2,2.0,2,10.35,고기를 굽는 장면


{'month': '2025-12',
 'mode': 'gpt',
 'month_overview': '12월 한 달 동안 아이는 따뜻한 가족의 일상 속에서 소소한 즐거움을 많이 경험했어요. 차 안에서 간식을 먹으며 편안한 시간을 보내고, 곰인형을 꼭 안아 사랑스러운 순간을 만들었습니다. 또한 고기를 굽는 모습을 통해 주변의 따스한 분위기를 느끼며 새로운 감각을 접하는 모습도 보였답니다.',
 'pattern_summary': '이번 달 활동 중 가장 자주 보인 모습은 식사와 간식 시간을 중심으로 한 가족 상호작용이었어요. 특히 차 안에서 간식을 먹는 장면이 여러 번 관찰되었고, 곰인형을 안는 행동도 자주 나타났지요. 놀이보다는 일상 생활 속 편안한 순간들을 즐기는 모습이 특징적입니다.',
 'change_summary': '이번 달에는 고기 굽기와 고기 냄새를 맡으며 식사하는 새로운 행동들이 나타났어요. 장난감 권총을 들고 상상 놀이를 하는 등 역할놀이 활동도 처음 시작되어 아이의 호기심과 상상력이 점차 확장되고 있음을 보여줍니다. 또한 손 잡기와 곰인형 안기 같은 교감 행동도 새롭게 시작되어 안정감과 친밀감이 깊어지는 시기임을 알 수 있습니다.',
 'parent_note': '아이에게 따뜻한 간식 시간과 곰인형을 안는 순간들이 얼마나 소중한지 느껴집니다. 집 안이나 차 안에서 느끼는 작은 행복들이 아이의 마음에 큰 안정을 주고 있으니, 앞으로도 이렇게 평범하지만 특별한 일상들을 함께 채워주세요. 새롭게 시작한 고기 굽기나 상상 놀이에도 관심과 응원을 보내주시면 아이의 경험이 더욱 풍부해질 거예요.',
 'one_line_summary': '12월, 아이는 가족과 함께하는 따뜻한 일상 속에서 새로운 감각과 놀이 경험을 시작하며 포근한 시간을 보냈습니다.',
 'highlights': [{'date': '2025-12-12',
   'text': '2025-12-12에는 차 안에서 간식 먹기 장면이 눈에 띄었다. 대표적으로는 차 안에서 간식 먹기 흐름으로 묶이